In [ ]:
import json
import csv
from collections import Counter
from transformers import pipeline

# -----------------------------
# 1. Load zero-shot classifier
# -----------------------------
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

LABELS = [
    "diagnostic or troubleshooting question",
    "action or operational request",
    "other informational question"
]

# -----------------------------
# 2. Load dataset
# -----------------------------
input_file = "/content/drive/MyDrive/QA/dataset_QA.json"

with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

# -----------------------------
# 3. Extract questions
# -----------------------------
questions = []

for entry in data.get("data", []):
    for para in entry.get("paragraphs", []):
        for qa in para.get("qas", []):

            question = qa.get("question", "").strip()

            if question:
                questions.append(question)

print(f"Total questions loaded: {len(questions)}")

# -----------------------------
# 4. Classify questions
# -----------------------------
def classify_question(question):

    result = classifier(
        question,
        candidate_labels=LABELS,
        hypothesis_template="This question is a {}."
    )

    label = result["labels"][0]
    confidence = result["scores"][0]

    return label, confidence


labels = []
confidences = []

for i, question in enumerate(questions):

    label, confidence = classify_question(question)

    labels.append(label)
    confidences.append(confidence)

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(questions)} questions")

# -----------------------------
# 5. Results
# -----------------------------
counts = Counter(labels)
total = len(questions)

print("\n=== Classification Results ===")

for label in LABELS:

    n = counts.get(label, 0)
    pct = (n / total * 100) if total else 0

    print(f"{label}: {n} ({pct:.2f}%)")

# -----------------------------
# 6. Examples
# -----------------------------
print("\n=== Sample Questions ===")

for category in LABELS:

    print(f"\n-- {category.upper()} --")

    examples = [
        (q, c)
        for q, l, c in zip(questions, labels, confidences)
        if l == category
    ][:5]

    for question, confidence in examples:
        print(f"  - [{confidence:.3f}] {question}")

# -----------------------------
# 7. Save results
# -----------------------------
output_csv = "erp_question_classification.csv"

with open(output_csv, "w", newline="", encoding="utf-8") as f:

    writer = csv.writer(f)

    writer.writerow([
        "question",
        "category",
        "confidence"
    ])

    for question, label, confidence in zip(
        questions,
        labels,
        confidences
    ):
        writer.writerow([
            question,
            label,
            confidence
        ])

print(f"\nSaved results to: {output_csv}")